# Initiation to data collection with web scraping and APIs


In [3]:
# Run this cell only if some packages are missing.
import importlib.util
import subprocess
import sys

packages = {
    "bs4": "beautifulsoup4",
    "pandas": "pandas",
    "lxml": "lxml",
    "folium": "folium",
}

for import_name, package_name in packages.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

print("All required packages are available.")

All required packages are available.


In [4]:
import bs4
import pandas as pd

from io import StringIO
from urllib.parse import quote, urljoin
from urllib.request import Request, urlopen


## Scrape a table on a Wikipedia page

We would like to display on a map the location of the Summer Olympic Games since 1896. We will use a [Wikipedia page](https://fr.wikipedia.org/wiki/Jeux_olympiques) to scrape the associated table.

Below is the code to download the page with `urllib.request` and display its title with `BeautifulSoup`.


In [5]:
url = "https://fr.wikipedia.org/wiki/Jeux_olympiques"

req = Request(url, headers={"User-Agent": "Mozilla/5.0"})
html = urlopen(req).read()

page = bs4.BeautifulSoup(html, "lxml")

print(page.find("title").get_text(strip=True))

Jeux olympiques — Wikipédia


Our objective here is to extract the information from the first table, "Jeux olympiques d'été", and build a dataframe.

To proceed, we will follow two steps:
* Find the table in the HTML page with Beautiful Soup.
* Convert the HTML table into a pandas dataframe.

Check the quick start of Beautiful Soup to understand what is going on: https://beautiful-soup-4.readthedocs.io/en/latest/


In [17]:
tables = page.find_all("table", class_="wikitable center")
#  title_ = "Jeux olympiques d'été"
# title_ = "Jeux olympiques d'hiver"
table_summer = tables[0]
table_winter = tables[1]

# pandas can read HTML tables directly.
olympics_summer = pd.read_html(StringIO(str(table_summer)))[0]
olympics_winter = pd.read_html(StringIO(str(table_winter)))[0]

# print('SUMMER')
# olympics_summer.head()
# olympics_summer.tail()
print("WINTER")
olympics_winter.head()


WINTER


,Année,N° (Nº de l’édition[NB 6]),Ville hôte,Pays,Continent
0,1924,I,Chamonix (1),France (1),Europe (1)
1,1928,II,Saint-Moritz (1),Suisse (1),Europe (2)
2,1932,III,Lake Placid (1),États-Unis (1),Amérique du Nord (1)
3,1936,IV,Garmisch-Partenkirchen (1),Allemagne (1),Europe (3)
4,1940,V (annulés[NB 7]),Sapporo,Japon,Asie


## Exercise 2: Locate places on a map

The objective here is to identify organizer cities on a map.

Here are the steps:
* For each city in the table, collect the URL of its corresponding page.
* Find the coordinates of the city in the HTML page of the city
* Store cities and coordinates in a dataframe.

Tips:
First, check manually how the page of a city is organized. With your favorite browser, check the source of the page and locate the code of the infobox on the topr right of the page. In that infobox, check where are the coordinates of the city and how this is coded in HTML. You will find that the coordinates are written in the HTML source at this location.

Read the following code and understand it line by line. Add comments to explain each of the parts.



In [20]:
base_url = "https://fr.wikipedia.org"
rows_summer = table_summer.find_all("tr") #list all rows with tr
# print(rows)
cities_summer = []
coordinates_cache_summer = {}

for row in rows_summer[1:]:
    cells = row.find_all("td") #list of all rows with td
    # print(cells)
    if len(cells) < 3:
        continue

    # BeautifulSoup can directly look for links with an href attribute.
    city_link = cells[2].find("a", href=True) #find ONE row with a (done for each row td)
    if city_link is None:
        continue
    # print(city_link)

    name = city_link.get_text(strip=True) #gets name of the city(ex Athenes, ...)
    # print(name) 
    city_url = urljoin(base_url, city_link["href"]) #as if we did "https://fr.wikipedia.org" + city_link["href"] = https://fr.wikipedia.org/wiki/Athènes
    #print(city_url) 

    if city_url not in coordinates_cache_summer:
        safe_city_url = quote(city_url, safe="/:?=&%")
        req = Request(safe_city_url, headers={"User-Agent": "Mozilla/5.0"})
        city_html = urlopen(req).read()
        city_page = bs4.BeautifulSoup(city_html, "lxml")

        map_link = city_page.select_one(
            "table[class*=infobox] a.mw-kartographer-maplink[data-lat][data-lon]"
        )

        if map_link is None:
            map_link = city_page.select_one("a.mw-kartographer-maplink[data-lat][data-lon]")

        coordinates_cache_summer[city_url] = (
            map_link.get("data-lat") if map_link else None,
            map_link.get("data-lon") if map_link else None,
        )

    latitude, longitude = coordinates_cache_summer[city_url]
    cities_summer.append({
        "name": name,
        "url": city_url,
        "latitude": latitude,
        "longitude": longitude,
    })

data_summer = pd.DataFrame(cities_summer).dropna()
data_summer["latitude"] = pd.to_numeric(data_summer["latitude"])
data_summer["longitude"] = pd.to_numeric(data_summer["longitude"])

data_summer.head()

,name,url,latitude,longitude
0,Athènes,https://fr.wikipedia.org/wiki/Athènes,37.966667,23.716667
1,Paris,https://fr.wikipedia.org/wiki/Paris,48.856578,2.351828
2,Saint-Louis,https://fr.wikipedia.org/wiki/Saint-Louis_(Mis...,38.648056,-90.212220
3,Athènes,https://fr.wikipedia.org/wiki/Athènes,37.966667,23.716667
4,Londres,https://fr.wikipedia.org/wiki/Londres,51.507222,-0.127500


In [ ]:
rows_winter = table_winter.find_all("tr") #list all rows with tr
# print(rows)
cities_winter = []
coordinates_cache_winter = {}

for row in rows_winter[1:]:
    cells = row.find_all("td") #list of all rows with td
    # print(cells)
    if len(cells) < 3:
        continue

    # BeautifulSoup can directly look for links with an href attribute.
    city_link = cells[2].find("a", href=True) #find ONE row with a (done for each row td)
    if city_link is None:
        continue
    # print(city_link)

    name = city_link.get_text(strip=True) #gets name of the city(ex Athenes, ...)
    # print(name) 
    city_url = urljoin(base_url, city_link["href"]) #as if we did "https://fr.wikipedia.org" + city_link["href"] = https://fr.wikipedia.org/wiki/Athènes
    #print(city_url) 

    if city_url not in coordinates_cache_winter:
        safe_city_url = quote(city_url, safe="/:?=&%")
        req = Request(safe_city_url, headers={"User-Agent": "Mozilla/5.0"})
        city_html = urlopen(req).read()
        city_page = bs4.BeautifulSoup(city_html, "lxml")

        map_link = city_page.select_one(
            "table[class*=infobox] a.mw-kartographer-maplink[data-lat][data-lon]"
        )

        if map_link is None:
            map_link = city_page.select_one("a.mw-kartographer-maplink[data-lat][data-lon]")

        coordinates_cache_winter[city_url] = (
            map_link.get("data-lat") if map_link else None,
            map_link.get("data-lon") if map_link else None,
        )

    latitude, longitude = coordinates_cache_winter[city_url]
    cities_winter.append({
        "name": name,
        "url": city_url,
        "latitude": latitude,
        "longitude": longitude,
    })

data_winter = pd.DataFrame(cities_winter).dropna() #supprimer tout ce qui est null
data_winter["latitude"] = pd.to_numeric(data_winter["latitude"]) #transformer la valeur string en numeric
data_winter["longitude"] = pd.to_numeric(data_winter["longitude"])

data_winter.head()

,name,url,latitude,longitude
0,Chamonix,https://fr.wikipedia.org/wiki/Chamonix-Mont-Blanc,45.923056,6.869722
1,Saint-Moritz,https://fr.wikipedia.org/wiki/Saint-Moritz,46.497777,9.838323
2,Lake Placid,https://fr.wikipedia.org/wiki/Lake_Placid_(New...,44.280600,-73.981390
3,Garmisch-Partenkirchen,https://fr.wikipedia.org/wiki/Garmisch-Partenk...,47.500000,11.100000
4,Sapporo,https://fr.wikipedia.org/wiki/Sapporo,43.050000,141.350000


Then we can plot the map. The map is also saved as an HTML file, so it can be opened in a browser if it does not display correctly inside the notebook.


In [33]:
from pathlib import Path

import folium
from IPython.display import display

center = data_summer[["latitude", "longitude"]].mean().tolist()

m = folium.Map(location=center, zoom_start=2, tiles="OpenStreetMap")

for _, city in data_summer.iterrows():
    folium.Marker(
        [city["latitude"], city["longitude"]],
        popup=city["name"],
    ).add_to(m)

sw = data_summer[["latitude", "longitude"]].min().tolist()
ne = data_summer[["latitude", "longitude"]].max().tolist()
m.fit_bounds([sw, ne])

Path("leaflet").mkdir(exist_ok=True)
m.save("leaflet/olympic_cities_map.html")

display(m)

## Questions
* Do the same for the Winter Olympic Games.
* Plot Summer and Winter Olympic cities on the same map, using a different color for each.


In [55]:
#WINTER OLYMPIC GAMES
center_summer = data_summer[["latitude", "longitude"]].mean().tolist()

m_sum_win = folium.Map(location=center_summer, zoom_start=2, tiles="OpenStreetMap")

for _, city in data_summer.iterrows():
    folium.Marker(
        [city["latitude"], city["longitude"]], #location= 
        popup=city["name"],
        icon=folium.Icon(color="green", icon="fa-leaf",  prefix="fa"),
    ).add_to(m_sum_win)
for _, city in data_winter.iterrows():
    folium.Marker(
        [city["latitude"], city["longitude"]],
        popup=city["name"],
        icon=folium.Icon(color="blue", icon="fa-thermometer-quarter",  prefix="fa"),
    ).add_to(m_sum_win)

sw = data_summer[["latitude", "longitude"]].min().tolist()
ne = data_summer[["latitude", "longitude"]].max().tolist()
m_sum_win.fit_bounds([sw, ne])

Path("leaflet").mkdir(exist_ok=True)
m_sum_win.save("leaflet/olympic_cities_summer_winter_map.html")

display(m_sum_win)

## Exercise 3: Use a REST API with OpenFoodFacts

In the previous exercises, we scraped HTML pages. Here we use a REST API: the server returns structured JSON directly.

We will use the [OpenFoodFacts API](https://openfoodfacts.github.io/documentation/). It is open and does not require an account for read requests. A good habit is to send a clear `User-Agent` header, so the service can identify educational requests if needed.

First check quickly the search documentation: https://openfoodfacts.github.io/documentation/docs/Product-Opener/v2/search/get-search/
Use this documentation to understand the code below.

Our mini mission: become snack detectives and find which chocolate products in the database look the sweetest.

Add comments to check that you understand everything that is going on.


In [ ]:
import json
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

api_url = "https://world.openfoodfacts.org/api/v2/search"

#https://openfoodfacts.github.io/openfoodfacts-server/api/tutorial-off-api/#describing-the-search-request
#https://openfoodfacts.github.io/openfoodfacts-server/api/ref-v2/#get-/api/v2/search

#on encode des paramtres de la requete
params = urlencode({
    "categories_tags_en": "chocolates",
    "page_size": 20,
    "fields": "code,product_name,brands,nutriscore_grade,nova_group,nutriments",
})

#on construit la requete
request = Request(
    f"{api_url}?{params}",
    headers={"User-Agent": "DALAS-course/1.0 (student exercise)"},
)
#on envoie la requete et on récupere le resultat
try:
    with urlopen(request) as response:
        results = json.load(response)
    print("Data loaded from OpenFoodFacts.")
except (HTTPError, URLError) as error:
    print("OpenFoodFacts is temporarily unavailable. Please run this cell again later.")
    print(error)
    results = {"products": []}

print("Number of products:", len(results["products"]))

Data loaded from OpenFoodFacts.
Number of products: 20


The response is a Python dictionary. The list of products is stored in `results["products"]`.

Each product is also a dictionary, and nutrition values are nested inside the `nutriments` dictionary.


In [ ]:
products = []

#On parcours le dictionnaire pour récuperer les produits
for product in results["products"]:
    nutriments = product.get("nutriments", {})
    code = product.get("code")

    products.append({
        "name": product.get("product_name"),
        "brand": product.get("brands"),
        "nutriscore": product.get("nutriscore_grade"),
        "nova_group": product.get("nova_group"),
        "sugars_100g": nutriments.get("sugars_100g"),
        "fat_100g": nutriments.get("fat_100g"),
        "energy_kcal_100g": nutriments.get("energy-kcal_100g"),
        "url": f"https://world.openfoodfacts.org/product/{code}" if code else None,
    })

columns = ["name", "brand", "nutriscore", "nova_group", "sugars_100g", "fat_100g", "energy_kcal_100g", "url"]
food = pd.DataFrame(products, columns=columns)
food.head()


,name,brand,nutriscore,nova_group,sugars_100g,fat_100g,energy_kcal_100g,url
0,Excellence Noir Prodigieux 90% Cacao,"Lindt, Lindt & Sprüngli",e,3.0,7.0,55.0,592.0,https://world.openfoodfacts.org/product/304692...
1,Noir Intense,"Lindt, LINDT SPRUNGLI SAS",e,3.0,30.0,41.0,566.0,https://world.openfoodfacts.org/product/304692...
2,Excellence 85% cacao,"Lindt, LINDT SPRUNGLI SAS",e,3.0,15.0,46.0,584.0,https://world.openfoodfacts.org/product/304692...
3,Шоколад 85% какаова маса,J. D. Gross,d,4.0,12.7,44.2,554.0,https://world.openfoodfacts.org/product/20995553
4,Excellence Noir Subtil Doux 70% Cacao,Lindt & Sprüngli,e,NaN,29.0,48.0,610.0,https://world.openfoodfacts.org/product/304692...


Now we can ask a small data question: among the returned products, which ones have the most sugar per 100g?


In [60]:
sweetest = (
    food.dropna(subset=["sugars_100g"])
    .sort_values("sugars_100g", ascending=False)
    .head(10)
)

sweetest[["name", "brand", "sugars_100g", "fat_100g", "nutriscore", "url"]]

,name,brand,sugars_100g,fat_100g,nutriscore,url
12,Chocolat au lait,"Milka, MONDELEZ EUROPE SERVICES GMBH",55.0,31.0,e,https://world.openfoodfacts.org/product/304514...
17,Excellence Noir à la Pointe de Fleur de Sel,Lindt EXCELLENCE,47.0,32.0,e,https://world.openfoodfacts.org/product/304692...
8,Nestlé dessert Noir,"Nestlé, NESTLE DESSERT, Tablettes",46.4,34.0,e,https://world.openfoodfacts.org/product/761303...
16,chocolat noir noisettes entieres,cote d\'or,40.0,39.0,e,https://world.openfoodfacts.org/product/762221...
10,Excellence 70% Cocoa Intense Dark,Lindt,30.0,41.0,e,https://world.openfoodfacts.org/product/304692...
1,Noir Intense,"Lindt, LINDT SPRUNGLI SAS",30.0,41.0,e,https://world.openfoodfacts.org/product/304692...
4,Excellence Noir Subtil Doux 70% Cacao,Lindt & Sprüngli,29.0,48.0,e,https://world.openfoodfacts.org/product/304692...
14,Buo Cioccolato Fondente 70%,,28.0,41.6,e,https://world.openfoodfacts.org/product/40896243
11,Cioccolato fondente,J. D. Gross (Lidl),27.2,40.0,e,https://world.openfoodfacts.org/product/20095291
7,Extra Dark 74% Cocoa,,26.0,42.0,e,https://world.openfoodfacts.org/product/20022464


### Questions

* Change `categories_tags_en` to another category: `breakfast-cereals`, `sodas`, `biscuits`, `ice-creams`, or `plant-based-milks`.
* Change `page_size` to get more products.
* Sort by another column, for example `fat_100g` or `energy_kcal_100g`.
* Add one extra field from the API response to the dataframe.
* Try to find the product that looks like the best snack: not too much sugar, not too much fat, and a decent Nutri-Score.


# Going further

* To get the full power of an API, you often have to read the documentation. For OpenFoodFacts: https://openfoodfacts.github.io/openfoodfacts-server/api/
* Try to use another API. Here is a list of public APIs on many topics: https://github.com/public-apis/public-apis
